# 小望遠鏡の実測データから 50cm 望遠鏡での写りを予測する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hsameshima/SmallTelescope_test01/blob/main/notebooks/m57_practicum.ipynb)

2026年10月8日にミューロン210（口径 210 mm）＋ ZWO ASI183MM で撮った M57（環状星雲）のデータから、同じカメラを国立天文台の 50cm 公開望遠鏡に付けたときの画素スケール・信号・S/N を予測し、本番の観測スケジュールを作ります。

**最初にやること**：メニューの「ファイル → ドライブにコピーを保存」で自分用のコピーを作ってください。コピーしないと作業内容が保存されません。

**進め方**：上のセルから順に実行します。`# TODO` と書かれたセルは自分でコードを書き、「答え」のセルには文章で答えます。提出は「ファイル → ダウンロード → .ipynb をダウンロード」で行います。

## 0. 準備（実行するだけ）

データをダウンロードし、解析に使う道具（関数）を読み込みます。セッションが切れたら、ここからやり直してください。

In [ ]:
import os
REPO = "SmallTelescope_test01"
if not os.path.exists(REPO):
    !git clone --depth 1 --branch main https://github.com/hsameshima/SmallTelescope_test01.git
DATA = os.path.join(REPO, "data", "20261008")
print(len(os.listdir(DATA)), "files in", DATA)

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.visualization import ZScaleInterval
from scipy import ndimage as ndi

plt.rcParams["figure.dpi"] = 100

def files(filt):
    """フィルター名（'Ha', 'O3', 'S2', 'R', 'G', 'B', 'Dark'）のファイル一覧"""
    return sorted(glob.glob(os.path.join(DATA, f"*filter-{filt}_*.fit")))

def load(path):
    """FITS を読み込んで float の 2 次元配列で返す（単位 ADU）"""
    return fits.getdata(path).astype(float)

def mad_std(a):
    """中央値絶対偏差から求めた標準偏差。星やホットピクセルに強い"""
    a = np.asarray(a, float)
    a = a[np.isfinite(a)]
    return 1.4826 * np.median(np.abs(a - np.median(a)))

def show(img, x=None, y=None, size=None, ax=None, title=None, marks=()):
    """画像を表示する（左下が原点）。x, y, size を与えるとその周りを拡大。
    marks に [(x, y), ...] を与えると円で印を付ける"""
    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 6))
    if size is None:
        vmin, vmax = ZScaleInterval().get_limits(img[np.isfinite(img)])
    else:
        cut = img[max(int(y - size), 0):int(y + size), max(int(x - size), 0):int(x + size)]
        vmin, vmax = np.nanpercentile(cut, [5, 99.5])
    ax.imshow(img, origin="lower", cmap="gray", vmin=vmin, vmax=vmax)
    if size is not None:
        ax.set_xlim(x - size, x + size); ax.set_ylim(y - size, y + size)
    for (mx, my) in marks:
        ax.plot(mx, my, "o", mfc="none", mec="tab:orange", ms=14, mew=1.5)
    ax.set_xlabel("x [pix]"); ax.set_ylabel("y [pix]")
    if title: ax.set_title(title)
    return ax

def centroid(img, x0, y0, r=8, n_iter=5):
    """(x0, y0) 付近にある星（または天体）の重心 (x, y) を返す"""
    yy, xx = np.mgrid[:img.shape[0], :img.shape[1]]
    x, y = float(x0), float(y0)
    for _ in range(n_iter):
        m = (xx - x)**2 + (yy - y)**2 <= r**2
        w = img[m] - np.median(img[(xx - x)**2 + (yy - y)**2 <= (2*r)**2])
        w = np.clip(w, 0, None)
        x, y = (xx[m] * w).sum() / w.sum(), (yy[m] * w).sum() / w.sum()
    return x, y

def aperture_mean(img, x, y, r=3):
    """(x, y) を中心とする半径 r 画素の円の中の平均値"""
    yy, xx = np.mgrid[:img.shape[0], :img.shape[1]]
    return np.nanmean(img[(xx - x)**2 + (yy - y)**2 <= r**2])

def sky_level(img, xc, yc, r_in=90, r_out=140):
    """(xc, yc) の周りのリング（半径 r_in〜r_out 画素）の中央値 = 背景のレベル"""
    yy, xx = np.mgrid[:img.shape[0], :img.shape[1]]
    d2 = (xx - xc)**2 + (yy - yc)**2
    return np.nanmedian(img[(d2 >= r_in**2) & (d2 <= r_out**2)])

def line_profile(img, x0, y0, angle_deg, rmax=80, width=2):
    """(x0, y0) を通り、x 軸から angle_deg 傾いた直線に沿った断面。
    r（-rmax〜+rmax 画素、0.25 画素刻み）と、幅 ±width 画素で平均した値を返す"""
    a = np.radians(angle_deg)
    r = np.arange(-rmax, rmax + 0.01, 0.25)
    prof = []
    for off in np.arange(-width, width + 0.01, 1.0):
        ys = y0 + r * np.sin(a) + off * np.cos(a)
        xs = x0 + r * np.cos(a) - off * np.sin(a)
        prof.append(ndi.map_coordinates(img, [ys, xs], order=1))
    return r, np.mean(prof, axis=0)

def _stars(img, nmax=40):
    d = img - ndi.median_filter(img, size=25)
    s = mad_std(d)
    lab, n = ndi.label(ndi.gaussian_filter(d, 1.5) > 8 * s)
    idx = np.arange(1, n + 1)
    area = ndi.sum(np.ones_like(d), lab, idx)
    flux = ndi.sum(d, lab, idx)
    com = np.array(ndi.center_of_mass(d, lab, idx))
    ok = area >= 6
    return com[ok][np.argsort(-flux[ok])[:nmax]]

def _shift(ref_stars, img):
    d = (ref_stars[:, None, :] - _stars(img)[None, :, :]).reshape(-1, 2)
    e = np.arange(-60.5, 61)
    H, _, _ = np.histogram2d(d[:, 0], d[:, 1], bins=[e, e])
    j, i = np.unravel_index(H.argmax(), H.shape)
    sel = (np.abs(d[:, 0] - (e[j] + .5)) < 1.5) & (np.abs(d[:, 1] - (e[i] + .5)) < 1.5)
    return np.median(d[sel], axis=0)

MASTER_DARK = np.mean([load(f) for f in files("Dark")], axis=0)
_REF_STARS = _stars(load(files("G")[0]) - MASTER_DARK)

def stack(filt):
    """そのフィルターの全フレームをダーク減算し、G バンド 1 枚目に位置を合わせて平均する。
    単位は ADU / 画素 / 60 秒（1 枚あたり）。端の欠けた部分は NaN"""
    out = []
    for f in files(filt):
        d = load(f) - MASTER_DARK
        dy, dx = _shift(_REF_STARS, d)
        out.append(ndi.shift(d, (dy, dx), order=3, mode="constant", cval=np.nan))
    return np.mean(out, axis=0)

print("準備完了")

## 課題1 画素スケールと視野

位置が分かっている星 2 個の間隔から画素スケールを求め、ミューロン210 の実際の焦点距離と、50cm での写り方を計算します。

| 星（Tycho-2） | 赤経 α（°） | 赤緯 δ（°） | V 等級 | 画像上のおよその位置 |
| --- | --- | --- | --- | --- |
| TYC 2642-84-1 | 283.26538 | +33.12722 | 10.5 | 右上の端付近（x ≈ 1230, y ≈ 905） |
| TYC 2643-1347-1 | 283.51846 | +32.88809 | 11.2 | 左下の隅付近（x ≈ 70, y ≈ 30） |

まず G バンドの 1 枚目を表示して、2 つの星を確かめます。

In [ ]:
g1 = load(files("G")[0]) - MASTER_DARK
print(g1.shape, "(縦, 横) 画素")
show(g1, title="G band, frame 1", marks=[(1230, 905), (70, 30)]);

### 1-1 星の重心を測る

`centroid(画像, x の目安, y の目安)` で重心を求めてください。拡大表示（`show(g1, x, y, size=20)`）で星の中心に印が来ているかも確かめます。

In [ ]:
# TODO: 2 つの星の重心 (x1, y1), (x2, y2) を求める
x1, y1 = ...
x2, y2 = ...
print(x1, y1, x2, y2)

### 1-2 角距離と画素スケール

2 星の角距離 θ は次の式で求まります（α, δ はラジアンに直して使う）。

$$\cos\theta = \sin\delta_1 \sin\delta_2 + \cos\delta_1 \cos\delta_2 \cos(\alpha_1 - \alpha_2)$$

θ を秒角（″）にし、画像上の距離（画素）で割ると画素スケール s（″/pix）になります。

In [ ]:
ra1, dec1 = 283.26538, 33.12722
ra2, dec2 = 283.51846, 32.88809

# TODO: 角距離 theta（秒角）、画像上の距離 d_pix（画素）、画素スケール s（″/pix）
theta = ...
d_pix = ...
s = ...
print(theta, d_pix, s)

### 1-3 焦点距離と F 値、50cm での画素スケールと視野

Bin4 の画素サイズは p = 2.4 µm × 4 = 9.6 µm です。焦点距離は f = 206265 × p / s で求まります。

50cm 公開望遠鏡は口径 500 mm、焦点距離 6030 mm です。画像の大きさは 1368 × 918 画素です。

In [ ]:
p_mm = 9.6e-3      # 画素サイズ [mm]
nx, ny = 1368, 918

# TODO: ミューロン210 の焦点距離 f_mew [mm] と F 値 F_mew（口径 210 mm）
f_mew = ...
F_mew = ...

# TODO: 50cm の画素スケール s_50 [″/pix] と F 値 F_50
s_50 = ...
F_50 = ...

# TODO: 両方の視野 [′]（横 × 縦）
fov_mew = ...
fov_50 = ...

**答え：**

（ここに書く）

### 1-4 M57 の外縁の大きさ

HST の観測では、M57 の電離ガスの外縁は長半径 約 42″、短半径 約 30″ です。

1. 42″ がミューロン210 と 50cm でそれぞれ何画素になるか計算してください。
2. Hα 画像で M57 の中心を求め、長軸に沿った断面（`line_profile`）を描いて、外側で強度がピークの半分に落ちる半径を読み取ってください。断面の角度を変えて、外縁がいちばん遠くなる向きが長軸です。

In [ ]:
ha = stack("Ha")
show(ha, 745, 445, 100, title="H-alpha (+[N II]), 2 x 60 s");

In [ ]:
# TODO: M57 の中心 (xc, yc) を求める（目で見た中心を目安に centroid(ha, x, y, r=60)。r はリング全体が入る大きさにする）
xc, yc = ...

# TODO: 42″ が何画素か（ミューロン210, 50cm）
r42_mew = ...
r42_50 = ...

# TODO: 角度を変えながら断面を描き、外側の半値半径を読み取る
r, prof = line_profile(ha, xc, yc, angle_deg=0)
plt.plot(r, prof); plt.xlabel("distance from centre [pix]"); plt.ylabel("ADU / pix / 60 s");

**答え：**

（ここに書く）

### 1-5 ビニングの検討

三鷹のシーイングを 4″ とします。50cm で星像が何画素に広がるか計算し、何画素ずつまとめる（ビニングする）のが適当か考えてください。

**答え：**

（ここに書く）

## 課題2 信号とノイズの実測

ミューロン210 での「60 秒・1 枚・1 画素あたり」の信号とノイズを実測します。

### 2-1 1 枚あたりのノイズ

同じ条件の 2 枚の差をとると、画素ごとに固定したパターン（ホットピクセルなど）が消え、ランダムなノイズだけが残ります。差の標準偏差は 1 枚のノイズの √2 倍です。ダークと各フィルターについて、1 枚のノイズ σ と、平均レベル（2 枚の平均の中央値）を求めてください。標準偏差には `mad_std` を使います。

In [ ]:
filters = ["Dark", "S2", "Ha", "O3", "R", "G", "B"]
level, sigma = {}, {}
for f in filters:
    a, b = [load(x) for x in files(f)]
    # TODO: 平均レベル level[f] と 1 枚のノイズ sigma[f]
    level[f] = ...
    sigma[f] = ...
    print(f, level[f], sigma[f])

### 2-2 ゲイン

光子（電子）の数 N はポアソン分布に従うので、分散は N に等しくなります。信号を ADU で表すと、ゲイン K（ADU / 電子）を使って分散は K × 平均 で増えます。横軸に平均レベル、縦軸に σ² をとって直線を当てはめ、傾きから K を求めてください。

In [ ]:
# TODO: level と sigma**2 をプロットし、np.polyfit で直線を当てはめて傾き K を求める
K = ...
print("K =", K, "ADU/e-,  1/K =", 1 / K, "e-/ADU")

**答え：**

（ここに書く）

### 2-3 M57 のリングの信号

各輝線の画像（2 枚を位置合わせして平均したもの）から背景を引き、リング上に半径 3 画素の開口を 4 か所以上置いて、平均の信号 S（ADU / pix / 60 s）とそのばらつきを求めてください。開口の位置は画像を拡大して自分で決めます。

In [ ]:
img = {f: stack(f) for f in ["Ha", "O3", "S2"]}
for f in img:
    img[f] = img[f] - sky_level(img[f], xc, yc)     # 背景を引く
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
for ax, f in zip(axs, img):
    show(img[f], xc, yc, 70, ax=ax, title=f)

In [ ]:
# TODO: リング上の開口の位置を決め、各輝線の平均信号 S[f] とばらつきを求める
positions = [(..., ...), (..., ...), (..., ...), (..., ...)]
S = {}
for f in img:
    vals = [aperture_mean(img[f], x, y, r=3) for x, y in positions]
    S[f] = np.mean(vals)
    print(f, S[f], np.std(vals))

### 2-4 1 枚・1 画素あたりの S/N

$$\left(\frac{S}{N}\right)_1 = \frac{S}{\sqrt{\sigma_{\rm bg}^2 + K S}}$$

σ_bg は 2-1 で求めたそのフィルターの 1 枚のノイズ、K S は天体そのものの光子ノイズ（ADU² 単位）です。

In [ ]:
# TODO: 各輝線の 1 枚・1 画素あたりの S/N
snr1_mew = {}

**答え：**

（ここに書く）

## 課題3 50cm での信号の換算

面輝度 Σ の天体を、口径 D・焦点距離 f の望遠鏡と画素サイズ p のカメラで撮るとき、1 画素が受ける光量は「集光面積」と「1 画素が見込む空の立体角」の積に比例します。

$$N_{\rm pix} \propto \Sigma \times \frac{\pi D^2}{4} \times \left(\frac{p}{f}\right)^2 \propto \Sigma\,\frac{p^2}{F^2}, \qquad F = \frac{f}{D}$$

1. 1 画素あたりの信号の比（50cm / ミューロン210）を計算してください。
2. 同じ角度面積（1″ × 1″）あたりの光子数の比を計算してください。
3. 星（点源）の総光量の比を計算してください。

In [ ]:
# TODO: 比を計算する
ratio_pix = ...     # 1 画素あたり
ratio_area = ...    # 同じ角度面積あたり
ratio_star = ...    # 星の総光量

**答え：**

（ここに書く）

## 課題4 S/N と必要積分時間の予測

カメラ・露出 60 秒・センサー温度は 10/8 と同じとし、背景ノイズ σ_bg は 1 画素あたり変わらないとします。S/N は枚数の平方根に比例して伸びます。

$$n_{\rm frame} = \left(\frac{(S/N)_{\rm goal}}{(S/N)_1}\right)^2$$

1. 50cm・Bin4 での 1 枚・1 画素あたりの S/N を各輝線で予測してください。
2. 3 × 3 画素にまとめた場合（約 1″/pix）の S/N を計算してください。まとめると信号は 9 倍、分散も 9 倍になります。
3. 3 色合成用に、リング上で 1 画素 S/N = 20 にするのに必要な枚数を、Bin4 のままの場合と 3 × 3 にまとめた場合で求めてください。
4. 動径分布用に、リングに沿った幅 1″ の楕円環で平均して S/N = 10 にするには何枚必要か見積もってください（環に含まれる画素数も自分で見積もる）。

In [ ]:
# TODO
S_50 = {}        # 50cm での信号 [ADU/pix/60s]
snr1_50 = {}     # 50cm・Bin4 の 1 枚・1 画素 S/N
snr1_50_b3 = {}  # 3x3 にまとめた場合
n_bin4 = {}      # S/N 20 に必要な枚数（Bin4）
n_b3 = {}        # S/N 20 に必要な枚数（3x3）

**答え：**

（ここに書く）

## 課題5 M57 の観測可能時間と当日のスケジュール

11月の M57 は夕方の西空にあり、薄明が終わるとまもなく低くなります。教員が指定する本番日について、使える時間を計算し、課題4の積分時間を割り付けたスケジュールを作ります。

| 計算に使う値 | |
| --- | --- |
| 観測地（国立天文台三鷹） | 北緯 35.672°、東経 139.538°、標高 58 m |
| M57 の位置（J2000） | α = 18h53m35.1s、δ = +33°01′45″ |
| 観測を許す高度の下限 | 30°（エアマス約 2） |

エアマスは X ≈ 1 / sin(高度) で近似できます。

In [ ]:
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import EarthLocation, SkyCoord, AltAz, get_sun, get_body
from astropy.utils import iers
iers.conf.auto_download = False          # 地球回転パラメータのダウンロードを省略（分単位の計算には影響しない）

OBS_DATE = "2026-11-11"                  # ← 教員が指定する本番日に書き換える
site = EarthLocation(lat=35.672 * u.deg, lon=139.538 * u.deg, height=58 * u.m)
m57 = SkyCoord("18h53m35.1s", "+33d01m45s")

jst_min = np.arange(16 * 60 + 30, 21 * 60 + 31, 1)                       # 16:30〜21:30 を 1 分刻み（日本時間）
times = Time(f"{OBS_DATE} 00:00:00") + (jst_min - 9 * 60) * u.min      # UTC に直す
frame = AltAz(obstime=times, location=site)
sun_alt = get_sun(times).transform_to(frame).alt.deg
m57_alt = m57.transform_to(frame).alt.deg

def hhmm(m):
    return f"{int(m // 60):02d}:{int(m % 60):02d}"
print("計算できました:", OBS_DATE)

In [ ]:
# TODO: 日没（太陽高度 -0.83°）、市民・航海・天文薄明の終了（-6°, -12°, -18°）の時刻
#       ヒント: sun_alt が初めて閾値を下回る位置を np.argmax(sun_alt < 閾値) で探し、jst_min から時刻を読む

# TODO: M57 の高度とエアマスをプロットし、天文薄明終了時の高度と、高度 30° を下回る時刻を求める

月の明るさと位置も確認します（天文薄明終了の時刻で計算）。輝面比は太陽と月の離角 φ から (1 − cos φ) / 2 で求まります。

In [ ]:
# TODO: 天文薄明終了時の月の輝面比・高度・M57 との離角
#       ヒント: moon = get_body("moon", times[i], site); sun = get_sun(times[i])

撮影の順番を決めるときは、次の 2 点を考えてください。

1. 高度が下がると大気減光が増え、減光は短い波長（[O III] 501 nm）ほど大きい
2. 輝線比を求めるには、フィルターごとのエアマスがそろっているほうがよい（交互に撮る方法もある）

フラットは空が明るいうちに、暗いナローバンドから撮ります。天候や機材のトラブルに備えて、予備時間も入れてください。

**答え：**

| 時刻 | 作業 | フィルター | 露出 × 枚数 | 確認すること |
| --- | --- | --- | --- | --- |
| | 機材の準備（カメラを外気になじませ、センサー温度を記録） | | | |
| （日没） | トワイライトフラット | | | |
| | 明るい星でピント合わせ | | | |
| | M57 の導入、試し撮り | | | |
| （天文薄明終了） | 本観測 | | | |
| （高度 30°） | 観測終了 | | | |
| | ダーク（同じ露出・同じ温度） | | | |

## 課題6 予測の限界

課題3・4の予測は、口径と F 値だけが違うという仮定に立っています。次の要因が、それぞれ予測をどちら向きにどの程度ずらすか、1・2 文で評価してください。評価した結果は、課題5のスケジュールの予備時間や確認項目に反映します。

- 副鏡による中央遮蔽と、鏡の反射率の違い
- センサー温度（11月の夜は 10/8 より気温が低く、ダーク電流が変わる）
- シーイングと追尾精度（画素が細かいほど、同じ角度のずれが多くの画素になる）
- M57 の高度と大気減光
- Hα フィルターへの [N II] の混入と、フィルター透過率の個体差
- 10/8 のデータはダーク 2 枚・フラットなしであること

**答え：**

（ここに書く）